<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/02_backends.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Files as the agent's workspace

In lesson 01 your agent wrote `report.md`. Nobody asked where that file went.

It turns out that question — **what outlives this conversation?** — is the most consequential
design decision in an agent, and it has a name: the **backend**.

**New in this lesson:** `StateBackend`, `StoreBackend`, `FilesystemBackend`, `CompositeBackend`,
and why the `execute` tool sometimes refuses to run.

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-02-backends"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. Why an agent needs a filesystem at all

Here is the problem, as a number.

Every tool result stays in the message history and is **re-sent on every subsequent model
call**. A 20,000-token search result is not a one-time cost — it is 20,000 tokens per step, for
the rest of the run.

In [ ]:
from langchain_core.messages.utils import count_tokens_approximately

# Pretend a tool returned a big document.
big_result = "The quarterly report states that revenue grew. " * 400
summary_line = "Wrote /notes/q3.md (12kb)"

per_step = count_tokens_approximately([{"role": "user", "content": big_result}])
offloaded = count_tokens_approximately([{"role": "user", "content": summary_line}])

print(f"one tool result:        ~{per_step:,} tokens")
print(f"carried for 20 steps:   ~{per_step * 20:,} tokens")
print()
print(f"as a file path instead: ~{offloaded:,} tokens per step "
      f"({offloaded * 20:,} over 20 steps)")

That is the whole argument. **Files let the agent pay for content once and re-read it
deliberately**, instead of dragging it through every future model call. This is called *context
offloading*, and it is why a filesystem ships in the harness rather than as an add-on.

The remaining question is where those files physically live.

---

## 2. The default: `StateBackend`

Files live in the agent's **graph state** — the same place its message list lives. Fast,
requires no setup, and lasts exactly as long as the conversation thread does.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=MODEL,
    system_prompt="You are a helpful assistant. Save any notes you take as files.",
)

result = agent.invoke({"messages": [{"role": "user", "content":
    "Write a two-line haiku about debugging and save it to haiku.md"
}]})

print(result["files"].get("haiku.md", "(no file)"))

In [ ]:
# Now a brand-new conversation. Same agent object.
second = agent.invoke({"messages": [{"role": "user", "content":
    "Read haiku.md and tell me what it says."
}]})

print(second["messages"][-1].text)

### 🧠 Checkpoint

The second call could not find `haiku.md`, even though it is the same `agent` object in the
same Python process. Where did the file actually live, and what does that tell you about what
"the agent" is?

<details><summary>Show answer</summary>

The file lived in the **state of the first invocation** — the dict that `invoke` returned. It
was never stored anywhere else.

`agent` is not a stateful object holding files. It is a compiled graph: a description of *how*
to process state. The state comes in as an argument and goes out as a return value. Two
`invoke` calls with no shared thread are two unrelated runs.

That is why the second call saw an empty filesystem. Everything in this lesson is about
choosing where state goes when you want it to outlive that.

</details>

---

## 3. Persisting across threads: `StoreBackend`

A `Store` is a key-value store that lives outside any single conversation. Point the backend at
one and files survive from thread to thread.

In [ ]:
from deepagents.backends import StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

persistent_agent = create_deep_agent(
    model=MODEL,
    system_prompt="You are a helpful assistant. Save any notes you take as files.",
    # A namespace keeps one agent's files from colliding with another's.
    # It is a callable, which is how you scope per-user in lesson 07.
    backend=StoreBackend(namespace=lambda rt: ("workshop", "files"), store=store),
)

persistent_agent.invoke({"messages": [{"role": "user", "content":
    "Write a two-line haiku about debugging and save it to haiku.md"
}]})

print("--- new thread, same store ---")
again = persistent_agent.invoke({"messages": [{"role": "user", "content":
    "Read haiku.md and tell me exactly what it says."
}]})
print(again["messages"][-1].text)

`InMemoryStore` still dies with this runtime — it is a store, not a database. Swap it for a
Postgres-backed store in production and the same code persists for years. **The agent does not
change; only where the store points does.**

---

## 4. Real files on a real disk: `FilesystemBackend`

Sometimes you want files you can actually look at — and that other programs can read.

In [ ]:
from deepagents.backends import FilesystemBackend

disk_agent = create_deep_agent(
    model=MODEL,
    system_prompt="You are a helpful assistant. Save any notes you take as files.",
    backend=FilesystemBackend(root_dir="workspace"),
)

disk_agent.invoke({"messages": [{"role": "user", "content":
    "Write a haiku about debugging to haiku.md"
}]})

In [ ]:
# These are ordinary files. Inspect them with ordinary tools.
!ls -la workspace
!cat workspace/haiku.md

### A backend also decides what the agent *can do*

The built-in `execute` tool runs shell commands — but only if the backend can actually execute
things. `StateBackend` is a dictionary; there is no shell in a dictionary.

In [ ]:
shell_task = {"messages": [{"role": "user", "content":
    "Run the shell command `echo hello from the sandbox` and tell me the output."
}]}

print("--- FilesystemBackend (has a real disk and shell) ---")
print(disk_agent.invoke(shell_task)["messages"][-1].text[:300])

print("\n--- StateBackend (no shell) ---")
print(agent.invoke(shell_task)["messages"][-1].text[:300])

---

## 5. Choosing

| Backend | Survives | Shell (`execute`) | Inspect with | Use when |
|---|---|---|---|---|
| `StateBackend` (default) | one thread | ✗ | the returned state dict | scratch space; the default is usually right |
| `StoreBackend` | forever, across threads | ✗ | store API | anything the agent should remember |
| `FilesystemBackend` | as long as the disk | ✓ | `ls`, `cat` | code execution, real artefacts, local dev |
| `CompositeBackend` | per route | depends on route | per route | you need more than one of the above at once |

Most real agents need **two** of these at once — which is what composite routing is for.

---

## 6. `CompositeBackend`: different paths, different homes

Route by path prefix. Here `/memories/` persists forever while everything else stays
thread-local scratch.

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend

routed_agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You are a helpful assistant.\n"
        "Durable facts about the user go in /memories/.\n"
        "Everything else is scratch work."
    ),
    backend=CompositeBackend(
        default=StateBackend(),
        routes={"/memories/": StoreBackend(namespace=lambda rt: ("workshop", "memories"), store=store)},
    ),
    store=store,
)

routed_agent.invoke({"messages": [{"role": "user", "content":
    "Remember that I prefer metric units — save it to /memories/preferences.md. "
    "Also jot a throwaway note to scratch.md."
}]})

print("--- new thread ---")
out = routed_agent.invoke({"messages": [{"role": "user", "content":
    "What do you know about my preferences? Also, does scratch.md still exist?"
}]})
print(out["messages"][-1].text)

### 🧠 Checkpoint

A customer support agent must remember a customer's tone preference forever, but must **not**
keep half-written draft replies between sessions.

Sketch the routing. Then say what breaks if you route everything to the store instead.

<details><summary>Show answer</summary>

```python
CompositeBackend(
    default=StateBackend(),                 # drafts, scratch work — die with the thread
    routes={"/memories/": StoreBackend(namespace=lambda rt: ("workshop", "memories"), store=store)},  # preferences — survive
)
```

If you route **everything** to the store, three things break:

1. **Cost and noise** — every abandoned draft is kept forever and shows up in future searches.
2. **Correctness** — the agent finds a half-written draft from three weeks ago and treats it as
   current intent.
3. **Privacy** — you are now indefinitely retaining content you never intended to keep, which is
   a compliance problem rather than a technical one.

"Persist everything" feels safe and is usually the wrong default. Persist what you would be
willing to *show the customer next year*.

</details>

### ✍️ Exercise

Build an agent with three routes:

- `/reports/` → real files on disk (`FilesystemBackend`)
- `/memories/` → the store, surviving threads
- everything else → thread-local scratch

Then prove all three behave differently: write to each in one thread, start a second thread,
and check what is still readable. Confirm the reports really exist on disk with `!ls`.

<details><summary>Show a solution</summary>

```python
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "Finished reports go in /reports/. Durable user facts go in /memories/. "
        "Everything else is scratch."
    ),
    backend=CompositeBackend(
        default=StateBackend(),
        routes={
            "/reports/": FilesystemBackend(root_dir="workspace"),
            "/memories/": StoreBackend(namespace=lambda rt: ("workshop", "memories"), store=store),
        },
    ),
    store=store,
)

agent.invoke({"messages": [{"role": "user", "content":
    "Write /reports/summary.md with one sentence about backends, "
    "/memories/note.md saying I like concise answers, and scratch.md with anything."
}]})

print(agent.invoke({"messages": [{"role": "user", "content":
    "Which of these can you still read: /reports/summary.md, /memories/note.md, scratch.md?"
}]})["messages"][-1].text)

!ls -R workspace
```

</details>

---

## 📌 Key takeaways

- The filesystem is a **context-management** tool: it keeps bulky content out of the message history so you pay for it once.
- "Which backend?" is really the question **"what should outlive this thread?"**
- `StateBackend` is thread-scoped, `StoreBackend` crosses threads, `FilesystemBackend` puts real files on a disk.
- Whether the `execute` tool works is a property of the **backend**, not the agent.
- `CompositeBackend` routes by path prefix, so one agent can have both scratch space and long-term memory.
- Persisting everything is not the safe default — it is a correctness and privacy problem.

---

## ➡️ Next

**[03 · Tools an agent can actually use](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/03_tools.ipynb)**

You have seen the tools that come free. Next you write your own — for a customer support agent —
and find out why a tool's **docstring** is one of the highest-leverage prompts in your codebase.